# Model C, Taken Apart: a Hands-On Primer
**Series B: Deep Learning for Molecules, Post 6g** &nbsp;|&nbsp; companion *primer* notebook

Model C (Shee, Li, Zhang, ..., Newhouse and Batista, *Nature Communications* **15**, 7818, 2024) is the **sampling** model of the family. Where Models A and B are deterministic (the diversity of their answers comes only from beam search), Model C is a **conditional kernel-elastic autoencoder (CKAE)**: it compresses a (template, center-labeled product) string into a 5,120-number **latent vector**, adds noise, and decodes it back *given the product*. At prediction time one draws latent vectors at random, so the same product yields many different templates; the latent space also gives a distance between reactions and lets one **interpolate** between two templates.

> **Two parts.** **Part I** needs only NumPy, pandas and matplotlib (about a minute). It rebuilds the CKAE at a small size and at the real size *analytically* (the parameter count 97,287,827 follows exactly from the architecture), analyzes the training loss and the latent noise, and works through the authors' saved outputs. **Part II** runs the authors' own PyTorch/RDKit code on the real Model C checkpoint, which is **not** in the capsule (the README gives a Google Drive link). **We could not run Part II** and, unlike for Models A and B, **we never ran Model C's weights** (we do not have them).
>
> Model C's checkpoint has 97 million parameters; everything about *its learned behavior* in this primer is therefore taken from the authors' saved outputs, not recomputed.

| Section | What you do |
|---|---|
| 1 | the CKAE data flow: shapes, the latent layer, the noise, the 10-vector memory |
| 2 | account for 97,287,827 parameters (where are they?) |
| 3 | the loss: reconstruction twice, plus a "modified MMD" (we compute it in closed form) |
| 4 | how much information can a noisy latent carry? |
| 5 | sampling from the prior: how the latent scale sets the diversity |
| 6 | the authors' saved outputs: funnel, cost, interpolation, Top-$K$ |
| **Part II** | run the authors' code with the downloaded checkpoint |

## 0. Setup

In [ ]:
import itertools, math, re, io, os, sys, pickle, zipfile, collections, time, textwrap, subprocess
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

def _as_code_dir(p):
    """p may be the capsule folder or its 'code' folder: return the folder that contains model_library, or None."""
    p = Path(p)
    for q in (p, p / "code"):
        if q.is_dir() and (q / "model_library").is_dir(): return q.resolve()
    return None

def _search(root, max_depth=3, deadline=None, want="model_library"):
    """Walk `root` (at most `max_depth` levels down, skipping hidden folders, stopping at `deadline`) for a folder that contains `want` (case-insensitive)."""
    root = Path(root)
    if not root.is_dir(): return None
    base = len(root.parts)
    for dirpath, dirnames, _ in os.walk(root):
        if deadline and time.time() > deadline: return None
        hit = next((d for d in dirnames if d.lower() == want), None)
        if hit: return Path(dirpath).resolve()
        if len(Path(dirpath).parts) - base >= max_depth: dirnames[:] = []
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
    return None

def _find_zip(root, max_depth, deadline):
    root = Path(root); base = len(root.parts)
    for dirpath, dirnames, files in os.walk(root):
        if time.time() > deadline: return None
        for f in files:
            if f.lower().startswith("capsule") and f.lower().endswith(".zip"): return Path(dirpath) / f
        if len(Path(dirpath).parts) - base >= max_depth: dirnames[:] = []
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
    return None

def find_capsule(mount_drive=True, max_depth=3, deep_seconds=90):
    """Find the uploaded Code Ocean capsule (the folder whose 'code' subfolder contains model_library).
    Order: environment variable CAPSULE_DIR; the working folder and /content; Google Drive (My Drive, 'Colab Notebooks');
    then a DEEP, time-limited search of Drive for the folder (or for capsule*.zip, which is unzipped into /content/capsule_unzipped).
    In Colab it offers to mount Google Drive if the capsule is not found elsewhere."""
    hit = _as_code_dir(os.environ["CAPSULE_DIR"]) if os.environ.get("CAPSULE_DIR") else None
    if hit: return hit
    def roots():
        r = [Path.cwd(), Path.cwd().parent, Path("/content")]
        for drive in ("/content/drive/MyDrive", "/content/drive/My Drive", "/content/gdrive/MyDrive"):
            r += [Path(drive), Path(drive) / "Colab Notebooks"]
        return [x for x in dict.fromkeys(r) if x.is_dir()]
    def look():
        for r in roots():                                                         # the usual places first (cheap)
            for q in (r / "capsule-3375825", r / "code", r):
                hit = _as_code_dir(q)
                if hit: return hit
        for r in roots():                                                         # a shallow search
            hit = _search(r, max_depth)
            if hit: return hit
        drives = [r for r in roots() if "drive" in str(r).lower() or "gdrive" in str(r).lower()]
        if drives:                                                                # a deep, time-limited search of Drive
            print(f"searching Google Drive more deeply (at most {deep_seconds} s) ...")
            deadline = time.time() + deep_seconds
            for r in drives[:1] + [d for d in drives[1:] if not str(d).startswith(str(drives[0]))]:
                hit = _search(r, 8, deadline)
                if hit: return hit
            for r in drives[:1]:
                z = _find_zip(r, 8, deadline)
                if z:
                    target = Path("/content/capsule_unzipped"); print("found", z, "-> unzipping into", target)
                    subprocess.run(["unzip", "-q", "-o", str(z), "-d", str(target)], check=False); hit = _search(target, 4)
                    if hit: return hit
        for r in roots():                                                         # a zip next to the notebook
            for z in sorted(r.glob("capsule-*.zip")):
                target = r / z.stem
                if not target.exists(): print("unzipping", z, "->", target); subprocess.run(["unzip", "-q", "-o", str(z), "-d", str(r)], check=False)
                hit = _search(r / z.stem, max_depth + 1) or _search(r, max_depth)
                if hit: return hit
        return None
    hit = look()
    if hit is None and mount_drive and not Path("/content/drive").exists():
        try:
            from google.colab import drive                                        # only exists in Colab; asks for your permission
            drive.mount("/content/drive"); hit = look()
        except Exception as e:
            print("(Google Drive was not mounted:", type(e).__name__, ")")
    if hit is None:                                                               # help the user see what Colab can see
        for r in roots():
            try: print(f"  {r}:", sorted(p.name for p in r.iterdir())[:20])
            except Exception as e: print(f"  {r}: cannot list ({type(e).__name__})")
    return hit

CAPSULE = find_capsule()
print("capsule found at:", CAPSULE if CAPSULE else "NOT FOUND: put the folder 'capsule-3375825' (or the zip) next to this notebook, in /content, or in My Drive / Colab Notebooks, or set CAPSULE_DIR (the real-weights cells will be skipped)")

In [ ]:
# ---- reading the authors' pickles WITHOUT executing anything: list what a pickle would import, then read it with a whitelist-only reader
import pickletools
def pickle_globals(data):
    strs, found = [], set()
    for op, arg, pos in pickletools.genops(data):
        if op.name in ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE"): strs.append(arg)
        if op.name == "GLOBAL": found.add(arg.replace(" ", "."))
        if op.name == "STACK_GLOBAL" and len(strs) >= 2: found.add(strs[-2] + "." + strs[-1])
    return sorted(found)

class _NoGlobals(pickle.Unpickler):
    def find_class(self, module, name): raise pickle.UnpicklingError(f"refused global {module}.{name}")
def load_plain_pickle(path):                                                           # for the small dictionary pickles (they import nothing)
    data = Path(path).read_bytes(); assert pickle_globals(data) == [], pickle_globals(data)
    return _NoGlobals(io.BytesIO(data)).load()

class _Storage:
    def __init__(self, key, numel): self.key, self.numel = key, numel
class _FloatStorage: pass
def _rebuild_tensor_v2(storage, offset, size, stride, requires_grad=False, backward_hooks=None, metadata=None): return ("tensor", storage, offset, tuple(size), tuple(stride))
class _CkptUnpickler(pickle.Unpickler):
    ALLOWED = {("collections", "OrderedDict"): collections.OrderedDict, ("torch", "FloatStorage"): _FloatStorage, ("torch._utils", "_rebuild_tensor_v2"): _rebuild_tensor_v2}
    def find_class(self, module, name):
        if (module, name) in self.ALLOWED: return self.ALLOWED[(module, name)]
        raise pickle.UnpicklingError(f"refused global {module}.{name}")
    def persistent_load(self, pid): return _Storage(str(pid[2]), int(pid[4]))

def load_checkpoint_numpy(path):
    """Read a PyTorch checkpoint (a zip with a pickle and raw tensor files) as NumPy arrays, importing only three standard tensor-storage names."""
    z = zipfile.ZipFile(path); pk = [n for n in z.namelist() if n.endswith("data.pkl")][0]; root = pk[:-len("data.pkl")]
    allowed = {"collections.OrderedDict", "torch.FloatStorage", "torch._utils._rebuild_tensor_v2"}
    unexpected = set(pickle_globals(z.read(pk))) - allowed
    assert not unexpected, f"the checkpoint imports something unexpected: {unexpected}"
    obj = _CkptUnpickler(io.BytesIO(z.read(pk))).load()
    def mat(o):
        if isinstance(o, tuple) and len(o) == 5 and o[0] == "tensor":
            _, st, off, size, stride = o; raw = np.frombuffer(z.read(root + "data/" + st.key), dtype="<f4")
            return np.lib.stride_tricks.as_strided(raw[off:], shape=size, strides=[s * 4 for s in stride]).copy() if len(size) else raw[off:off + 1].copy()
        if isinstance(o, dict): return type(o)((k, mat(v)) for k, v in o.items())
        if isinstance(o, (list, tuple)): return type(o)(mat(v) for v in o)
        return o
    return mat(obj)
print("safe readers defined")

In [ ]:
# ---- the tokenizer: characters, with two-letter atoms merged and '>>' merged; <SOS> first, '?' last (our re-implementation of the capsule's 'character_consider_atoms')
TWO_LETTER_ATOMS = ["Ag","Al","As","Bi","Br","Ca","Cl","Cr","Cu","Ga","Ge","Hg","Li","Mg","Mn","Na","Pb","Pt","Ru","Sb","Se","Si","Sn","Ta","Te","Ti","Tl","Xe","Zn","Zr","se","si","te","Mo","Re"]
def tokenize(s):
    toks, has_added, skip = ["<SOS>"], False, False
    for i, c in enumerate(s):
        if skip: skip = False; continue
        if i < len(s) - 1 and c + s[i + 1] in TWO_LETTER_ATOMS: skip = True; toks.append(c + s[i + 1])
        elif c != ">": toks.append(c)
        elif not has_added: toks.append(">>"); has_added = True
    return toks + ["?"]

# ---- atoms in a SMILES string, in RDKit's numbering (the order of appearance), without RDKit
def smiles_atom_spans(smiles):
    spans, i, n = [], 0, len(smiles)
    while i < n:
        c = smiles[i]
        if c == "[":
            j = smiles.index("]", i); k = i + 1
            while k < j and smiles[k].isdigit(): k += 1                                   # isotope
            m = re.match(r"(Cl|Br|Si|Mg|Mn|Li|Cr|Cu|Zn|Al|Na|Ti|Ge|Zr|Bi|Sb|Te|As|Mo|Ga|Hg|Ta|Pb|Ru|Pt|Xe|Tl|Re|Ag|Ca|Sn|Se|se|si|te|[A-Za-z*])", smiles[k:j])
            spans.append((k, k + len(m.group(1)))); i = j + 1
        elif smiles[i:i + 2] in ("Cl", "Br"): spans.append((i, i + 2)); i += 2
        elif c in "BCNOPSFIbcnops*": spans.append((i, i + 1)); i += 1
        elif c == "%": i += 3
        else: i += 1                                                                   # bonds, branches, ring digits, '.', stereo marks
    return spans

def to_center_labeled(smiles, centers):
    """The center-labeled product: the element symbol of every atom whose index is in `centers` is replaced by '*'."""
    spans = smiles_atom_spans(smiles); out, last = [], 0
    for idx, (a, b) in enumerate(spans):
        out.append(smiles[last:a]); out.append("*" if idx in centers else smiles[a:b]); last = b
    out.append(smiles[last:]); return "".join(out)

def centers_of(center_labeled):
    return [i for i, (a, b) in enumerate(smiles_atom_spans(center_labeled)) if center_labeled[a:b] == "*"]

def star_positions(center_labeled):
    """Positions of the '*' tokens in the token sequence (position 0 is <SOS>): what Model B adds a '*' embedding at."""
    return [i for i, t in enumerate(tokenize(center_labeled)) if t == "*"]

def template_anatomy(template):
    """Split a retro template 'product side >> reactant side' into its parts."""
    left, right = template.split(">>"); maps = lambda s: re.findall(r":(\d+)\]", s)
    return dict(product_side=left, reactant_side=right, mapped_atoms=len(set(maps(left))), unmapped_new_atoms=len(re.findall(r"\[[^\]:]+\]|Br|Cl|[BCNOPSFI]", re.sub(r"\[[^\]]*:\d+\]", "", right))),
                reactant_fragments=len(right.split(".")), product_fragments=len(left.split(".")), intramolecular=len(left.split(".")) > 1)

# ---- tests against center-labeled products that the authors saved in their notebooks
assert to_center_labeled("C#CC1(C)CCCCC1=O", [0, 1]) == "*#*C1(C)CCCCC1=O"
_p = "CC(=O)OCC(=O)[C@@]1(O)CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@]4(C)C3=CC[C@]12C"
assert to_center_labeled(_p, [5, 6, 4, 3, 1, 2]) == "C*(=*)***(=*)[C@@]1(O)CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@]4(C)C3=CC[C@]12C"
_q, _clp = "CC(=O)c1ccc(Cn2ncc(NC(=O)c3nc(C)oc3-c3cccc(C(F)(F)F)c3)n2)o1", "C*(=*)c1ccc(Cn2ncc(**(=O)c3nc(C)oc3-c3cccc(C(F)(F)F)c3)n2)o1"
assert to_center_labeled(_q, centers_of(_clp)) == _clp and centers_of(_clp) == [1, 2, 12, 13]
assert star_positions("*#*C1(C)CCCCC1=O") == [1, 3]
print("the string-level atom tools reproduce the center-labeled products saved in the authors' notebooks")

## 1. The data flow of the CKAE
Training sees pairs (a template-plus-center-labeled-product string $s$, its product $p$). In the capsule's code:
1. A **condition encoder** (6 layers) reads the product tokens $p$ (up to 310) and a linear layer **contracts the 310 positions to 10 vectors**: the *condition* $c$ (10 vectors of size 512).
2. The **main encoder** (6 layers) reads the embeddings of $s$ (up to 350 tokens) **with the 10 condition vectors appended**: 360 positions.
3. A linear layer maps this to the **latent** $z$: 10 positions $\times$ 512 $=$ **5,120 numbers**.
4. During training, noise is added: $\tilde z=z+\varepsilon$, $\varepsilon\sim N(0,I)$.
5. $\tilde z$ is concatenated with the condition ($5120+5120=10{,}240$ numbers) and a linear layer **`inverse`** maps it to 10 vectors of size 512: the **memory** of the decoder.
6. The decoder (6 layers) writes $s$ back, attending to those 10 vectors only (no mask).

At prediction time steps 1 to 4 are replaced by **drawing $z\sim N(0,I)$** (5,120 numbers); the condition still comes from the product.
We build the network at a **small size** (width 32, 2 layers) with the capsule's parameter names and run it; then we count the real size.
**Stop and predict:** after the `inverse` layer, how many positions does the decoder attend to?

In [ ]:
def softmax(x, ax=-1):
    e = np.exp(x - x.max(ax, keepdims=True)); return e / e.sum(ax, keepdims=True)
def layer_norm(x, w, b, eps=1e-5):
    m = x.mean(-1, keepdims=True); v = x.var(-1, keepdims=True); return (x - m) / np.sqrt(v + eps) * w + b
def linear(x, W, b): return x @ W.T + b

def ckae_param_count(d, ff, layers, vocab, enc_len, prod_len, cond_len, bottle_len):
    """Parameters of the capsule's Model C, from the architecture. enc_len: length of the template string slots (350); prod_len: product slots (310)."""
    ln, mha, fnn = 2 * d, 4 * (d * d + d), d * ff + ff + ff * d + d
    stack = lambda: layers * (2 * ln + mha + fnn)
    main_enc = vocab * d + enc_len * d + stack()
    cond_enc = vocab * d + prod_len * d + stack()
    contraction = prod_len * cond_len + cond_len
    latent = (enc_len + cond_len) * bottle_len + bottle_len
    inverse = (bottle_len + cond_len) * d * (bottle_len * d) + bottle_len * d
    dec = vocab * d + enc_len * d + layers * (3 * ln + 2 * mha + fnn) + d * vocab + vocab
    return dict(main_encoder=main_enc, condition_encoder=cond_enc, contraction=contraction, latent=latent, inverse=inverse, decoder=dec)

real = ckae_param_count(d=512, ff=1024, layers=6, vocab=83, enc_len=350, prod_len=310, cond_len=10, bottle_len=10)
tot = sum(real.values()); print(f"Model C at its real size: {tot:,} parameters; the authors' notebooks print 97,287,827"); assert tot == 97_287_827

In [ ]:
# a small CKAE in NumPy with the capsule's structure (random weights); names follow the capsule's modules
def build_small(d=32, ff=64, layers=2, vocab=83, enc_len=40, prod_len=30, cond_len=4, bottle_len=5, seed=0):
    r = np.random.default_rng(seed); W = lambda *s: (r.normal(size=s) * 0.08).astype(np.float32); sd = {}
    def attn(p):
        for n in "qkvo": sd[f"{p}.fc_{n}.weight"], sd[f"{p}.fc_{n}.bias"] = W(d, d), W(d)
    def ffn(p): sd[p + ".fc_1.weight"], sd[p + ".fc_1.bias"], sd[p + ".fc_2.weight"], sd[p + ".fc_2.bias"] = W(ff, d), W(ff), W(d, ff), W(d)
    def norm(p): sd[p + ".weight"], sd[p + ".bias"] = np.ones(d, np.float32), np.zeros(d, np.float32)
    def enc(prefix, length):
        sd[prefix + ".tok_embedding.weight"], sd[prefix + ".pos_embedding.weight"] = W(vocab, d), W(length, d)
        for l in range(layers):
            p = f"{prefix}.layers.{l}"; attn(p + ".self_attention"); ffn(p + ".positionwise_feedforward"); norm(p + ".self_attn_layer_norm"); norm(p + ".ff_layer_norm")
    enc("encoder", enc_len); enc("conditioner.encoder", prod_len)
    sd["conditioner.sequence_contraction.weight"], sd["conditioner.sequence_contraction.bias"] = W(cond_len, prod_len), W(cond_len)
    sd["latent.weight"], sd["latent.bias"] = W(bottle_len, enc_len + cond_len), W(bottle_len)
    sd["inverse.weight"], sd["inverse.bias"] = W(bottle_len * d, (bottle_len + cond_len) * d), W(bottle_len * d)
    sd["decoder.tok_embedding.weight"], sd["decoder.pos_embedding.weight"] = W(vocab, d), W(enc_len, d)
    for l in range(layers):
        p = f"decoder.layers.{l}"; attn(p + ".self_attention"); attn(p + ".encoder_attention"); ffn(p + ".positionwise_feedforward")
        for n in ("self_attn_layer_norm", "enc_attn_layer_norm", "ff_layer_norm"): norm(f"{p}.{n}")
    sd["decoder.fc_out.weight"], sd["decoder.fc_out.bias"] = W(vocab, d), W(vocab)
    return sd, dict(d=d, layers=layers, heads=2, vocab=vocab, enc_len=enc_len, prod_len=prod_len, cond_len=cond_len, bottle_len=bottle_len, pad=vocab - 1)

class CKAE_NP:
    def __init__(self, sd, cfg): self.sd, self.c = sd, cfg
    def mha(self, p, q, kv, keep):
        sd, d, h = self.sd, self.c["d"], self.c["heads"]; dh = d // h; B, Lq, _ = q.shape; Lk = kv.shape[1]
        Q = linear(q, sd[p + ".fc_q.weight"], sd[p + ".fc_q.bias"]).reshape(B, Lq, h, dh).transpose(0, 2, 1, 3)
        K = linear(kv, sd[p + ".fc_k.weight"], sd[p + ".fc_k.bias"]).reshape(B, Lk, h, dh).transpose(0, 2, 1, 3)
        V = linear(kv, sd[p + ".fc_v.weight"], sd[p + ".fc_v.bias"]).reshape(B, Lk, h, dh).transpose(0, 2, 1, 3)
        a = softmax(np.where(keep, Q @ K.transpose(0, 1, 3, 2) / math.sqrt(dh), -1e10))
        return linear((a @ V).transpose(0, 2, 1, 3).reshape(B, Lq, d), sd[p + ".fc_o.weight"], sd[p + ".fc_o.bias"])
    def ffn(self, p, x): sd = self.sd; return linear(np.maximum(linear(x, sd[p + ".fc_1.weight"], sd[p + ".fc_1.bias"]), 0), sd[p + ".fc_2.weight"], sd[p + ".fc_2.bias"])
    def encoder(self, prefix, tokens, keep, condition=None):
        sd, d = self.sd, self.c["d"]; B, L = tokens.shape; x = sd[prefix + ".tok_embedding.weight"][tokens] * math.sqrt(d) + sd[prefix + ".pos_embedding.weight"][np.arange(L)][None]
        if condition is not None: x = np.concatenate([x, condition], axis=1)                                     # the condition is appended AFTER embedding
        for l in range(self.c["layers"]):
            p = f"{prefix}.layers.{l}"; x = layer_norm(x + self.mha(p + ".self_attention", x, x, keep), sd[p + ".self_attn_layer_norm.weight"], sd[p + ".self_attn_layer_norm.bias"])
            x = layer_norm(x + self.ffn(p + ".positionwise_feedforward", x), sd[p + ".ff_layer_norm.weight"], sd[p + ".ff_layer_norm.bias"])
        return x
    def condition(self, product):                                                                                  # (B, prod_len) -> (B, cond_len, d)
        c, sd = self.c, self.sd; keep = (product != c["pad"])[:, None, None, :]; e = self.encoder("conditioner.encoder", product, keep)
        return linear(e.transpose(0, 2, 1), sd["conditioner.sequence_contraction.weight"], sd["conditioner.sequence_contraction.bias"]).transpose(0, 2, 1)
    def latent(self, src, condition):
        c, sd = self.c, self.sd; B = src.shape[0]; keep = np.concatenate([(src != c["pad"]), np.ones((B, c["cond_len"]), bool)], 1)[:, None, None, :]
        e = self.encoder("encoder", src, keep, condition)                                                          # (B, enc_len + cond_len, d)
        z = linear(e.reshape(B, c["d"], c["enc_len"] + c["cond_len"]), sd["latent.weight"], sd["latent.bias"])     # the capsule RESHAPES (views) the output as (d, positions): see the note below
        return z.reshape(B, c["bottle_len"] * c["d"]), e
    def memory(self, z, condition):                                                                                # latent + condition -> the decoder's memory (B, bottle_len, d)
        c, sd = self.c, self.sd; B = z.shape[0]; zc = np.concatenate([z, condition.reshape(B, -1)], axis=1)
        return linear(zc, sd["inverse.weight"], sd["inverse.bias"]).reshape(B, c["d"], c["bottle_len"]).transpose(0, 2, 1)
    def decode(self, trg, memory):
        c, sd = self.c, self.sd; d = c["d"]; B, t = trg.shape
        x = sd["decoder.tok_embedding.weight"][trg] * math.sqrt(d) + sd["decoder.pos_embedding.weight"][np.arange(t)][None]
        keep = (trg != c["pad"])[:, None, None, :] & np.tril(np.ones((t, t), bool))[None, None]; mkeep = np.ones((B, 1, 1, memory.shape[1]), bool)       # no mask on the memory
        for l in range(c["layers"]):
            p = f"decoder.layers.{l}"; x = layer_norm(x + self.mha(p + ".self_attention", x, x, keep), sd[p + ".self_attn_layer_norm.weight"], sd[p + ".self_attn_layer_norm.bias"])
            x = layer_norm(x + self.mha(p + ".encoder_attention", x, memory, mkeep), sd[p + ".enc_attn_layer_norm.weight"], sd[p + ".enc_attn_layer_norm.bias"])
            x = layer_norm(x + self.ffn(p + ".positionwise_feedforward", x), sd[p + ".ff_layer_norm.weight"], sd[p + ".ff_layer_norm.bias"])
        return linear(x, sd["decoder.fc_out.weight"], sd["decoder.fc_out.bias"])

sd_s, cfg = build_small(); n_small = sum(v.size for v in sd_s.values())
formula_small = sum(ckae_param_count(d=cfg["d"], ff=64, layers=cfg["layers"], vocab=cfg["vocab"], enc_len=cfg["enc_len"], prod_len=cfg["prod_len"], cond_len=cfg["cond_len"], bottle_len=cfg["bottle_len"]).values())
print(f"small CKAE: {n_small:,} parameters; formula {formula_small:,}"); assert n_small == formula_small
M = CKAE_NP(sd_s, cfg); rg = np.random.default_rng(3); pad = cfg["pad"]
product = np.array([list(rg.integers(0, 82, 12)) + [pad] * (cfg["prod_len"] - 12)]); src = np.array([list(rg.integers(0, 82, 25)) + [pad] * (cfg["enc_len"] - 25)]); trg = rg.integers(0, 82, (1, 15))
cond = M.condition(product); z, enc_out = M.latent(src, cond); noisy = z + rg.normal(size=z.shape); mem_noisy, mem_clean = M.memory(noisy, cond), M.memory(z, cond)
print("condition", cond.shape, "| encoder output", enc_out.shape, "| latent", z.shape, f"(= {cfg['bottle_len']} positions x {cfg['d']})", "| decoder memory", mem_clean.shape, "| logits", M.decode(trg, mem_clean).shape)
print(f"the decoder attends to {mem_clean.shape[1]} vectors (bottle_len = {cfg['bottle_len']}): the whole template must pass through this memory")
print(f"noisy and clean latents give different logits: max difference {np.abs(M.decode(trg, mem_noisy) - M.decode(trg, mem_clean)).max():.2f}")
assert mem_clean.shape == (1, cfg["bottle_len"], cfg["d"])

In [ ]:
# a detail of the capsule's code: the encoder output (positions x channels) is VIEWED as (channels x positions) before the latent layer, not transposed.
e = enc_out[0]; d, Lp = cfg["d"], cfg["enc_len"] + cfg["cond_len"]; W = sd_s["latent.weight"]
viewed = e.reshape(d, Lp) @ W.T; transposed = e.T @ W.T                                           # (d, bottle_len) each
print(f"'view' (as in the capsule) and a true transpose give different latents (max difference {np.abs(viewed - transposed).max():.2f}):")
print("with the view, each row mixes", Lp, "CONSECUTIVE numbers of the position-major layout, i.e. pieces of neighboring positions and channels; with a transpose each row would be one channel along all positions.")
print("both are linear maps and can be trained; the checkpoint was trained with the view (our reading of the code; we cannot tell whether it was intended).")

## 2. Where are the 97 million parameters?

In [ ]:
rows = [(k, f"{v:,}", f"{100 * v / tot:.1f} %") for k, v in sorted(real.items(), key=lambda kv: -kv[1])]
print(pd.DataFrame(rows, columns=["component", "parameters", "share"]).to_string(index=False))
print(f"\nthe `inverse` layer alone ({real['inverse']:,}) holds {100 * real['inverse'] / tot:.0f} % of the model: it maps (5,120 latent + 5,120 condition) numbers to 5,120 numbers.")
print(f"the three Transformer stacks together (two encoders and the decoder) have {real['main_encoder'] + real['condition_encoder'] + real['decoder']:,} = {100 * (real['main_encoder'] + real['condition_encoder'] + real['decoder']) / tot:.0f} %; the same Transformer at width 256 (Model A) has 8.1 million in total.")
# the latent bottleneck, in numbers: what can pass from the encoder to the decoder?
print(f"\nencoder output: 360 positions x 512 = {360 * 512:,} numbers; latent: 10 x 512 = {10 * 512:,}; decoder memory: 10 x 512; compression of the encoder output {360 * 512 / 5120:.0f}:1")
assert abs(real["inverse"] / tot - 0.539) < 0.001

## 3. The loss: two reconstructions and a "modified MMD"
With $\lambda=1$ and $\delta=0$ the capsule's loss for a batch is
$$\mathcal L=\lambda\,\mathrm{mMMD}(z)+\frac{\mathcal L_{\rm rec}(z+\varepsilon)+(\lambda+\delta)\,\mathcal L_{\rm rec}(z)}{\lambda+\delta+1}=\mathrm{mMMD}(z)+\tfrac12\Big[\mathcal L_{\rm rec}(z+\varepsilon)+\mathcal L_{\rm rec}(z)\Big],$$
the average of the cross-entropy of reconstructing the string from the **noisy** latent and from the **clean** latent. The regularizer compares the batch's latents $z$ with 1000 fresh draws $y\sim N(0,I_{5120})$ through a Gaussian kernel $K(x,y)=\exp\!\big(-\tfrac1D\sum_d(x_d-y_d)^2/0.64\big)$ and keeps **only the cross term**: $\mathrm{mMMD}=1-\overline{K(z,y)}$. (A full MMD would also contain $\overline{K(z,z')}$ and $\overline{K(y,y')}$.)
**Stop and predict:** which latent minimizes the average cross-kernel penalty?

In [ ]:
D, bw = 5120, 0.64
def K(x, y): return np.exp(-np.mean((x[:, None, :] - y[None, :, :]) ** 2, axis=2) / bw)       # the capsule's kernel (the squared distance is averaged over the D dimensions)
s = 1 / (bw * D)
def expected_K(norm2): return (1 + 2 * s) ** (-D / 2) * math.exp(-s * norm2 / (1 + 2 * s))   # E_y K(x, y) for y ~ N(0, I_D): a function of |x|^2 only
rg = np.random.default_rng(0); Y = rg.normal(size=(1000, D)).astype(np.float64)
rows = []
for sigma_z in (0.0, 0.05, 0.1, 0.3, 1.0, 3.0):
    x = rg.normal(size=(1, D)) * sigma_z; n2 = float((x ** 2).sum()); mc = float(K(x, Y).mean())
    rows.append((sigma_z, f"{n2:,.0f}", f"{mc:.4f}", f"{expected_K(n2):.4f}", f"{1 - expected_K(n2):.4f}"))
print(pd.DataFrame(rows, columns=["latent entry scale", "|z|^2", "mean kernel (Monte Carlo)", "closed form", "mMMD = 1 - kernel"]).to_string(index=False))
print(f"\nthe penalty depends on |z|^2 only, grows from {1 - expected_K(0):.3f} at the origin towards 1, and is almost FLAT for small latents: from the origin to entries of scale 0.1 (|z|^2 ~ 51) it changes by {expected_K(0) - expected_K(51):.4f}.")
print(f"its gradient pulls z towards the origin with strength {2 * s / (1 + 2 * s) * expected_K(0):.1e} x z per component: a very weak regularizer at this bandwidth (our analysis of the capsule's formula).")
assert abs(expected_K(0) - 0.2097) < 0.001 and all(abs(float(r[2]) - float(r[3])) < 0.01 for r in rows)

## 4. How much information can a noisy latent carry?
The decoder never sees $z$ itself during sampling; during training it sees $z+\varepsilon$ with unit-variance noise in every one of the 5,120 dimensions. If the encoder's latent entries are small (the authors' interpolation notebook prints latent entries of about $\pm0.1$), the signal-to-noise ratio per dimension is tiny, but there are many dimensions. A Gaussian channel with signal variance $\sigma_z^2$ and noise variance 1 carries at most
$$I\le\frac D2\log_2\!\big(1+\sigma_z^2\big)\ \text{bits}.$$

In [ ]:
rows = [(sz, f"{D / 2 * math.log2(1 + sz ** 2):,.1f}", f"10^{D / 2 * math.log2(1 + sz ** 2) * math.log10(2):,.1f}") for sz in (0.03, 0.05, 0.1, 0.2, 0.5, 1.0)]
print(pd.DataFrame(rows, columns=["latent entry scale", "capacity bound (bits)", "distinguishable messages (2^bits)"]).to_string(index=False))
print("\nat the scale of about 0.1 that the authors' printout suggests (it shows only the first and last 3 of 5,120 entries per vector), the latent can carry at most ~37 bits: about 10^11 distinguishable messages,")
print("far more than the thousands of templates that matter for one product (about 13 bits for 10^4): the noise limits how FINE the latent code can be, not whether it can index a template.")
assert abs(D / 2 * math.log2(1 + 0.1 ** 2) - 36.7) < 0.1

## 5. Sampling from the prior: how the latent scale sets the diversity
At prediction time $z\sim N(0,I)$ replaces the encoder. What templates come out? Here is a toy with the *exact* Bayes decoder for a simple code: $K$ "templates" with training frequencies $\pi_k$, each encoded as a latent mean $\mu_k=r\,u_k$ (random unit directions $u_k$ in $D=5120$ dimensions, scale $r$); the decoder sees $\mu_k+\varepsilon$ and returns the exact posterior $p(k\mid\tilde z)\propto\pi_k\exp(\mu_k\!\cdot\!\tilde z-r^2/2)$. We **sample** $z\sim N(0,I)$ and count the templates the decoder chooses, once with greedy decoding (the most probable template for each draw) and once by sampling from the posterior.
**Stop and predict:** with greedy decoding (what beam search approximates), what happens at $r=0$, and what at large $r$?

In [ ]:
pi = np.array([0.50, 0.25, 0.12, 0.08, 0.03, 0.02]); Kc = len(pi); rg = np.random.default_rng(5); U = rg.normal(size=(Kc, D)); U /= np.linalg.norm(U, axis=1, keepdims=True)
Zs = rg.normal(size=(4000, D)); H = lambda f: float(-(f[f > 0] * np.log2(f[f > 0])).sum())              # prior draws; entropy in bits
rows, ent_greedy, ent_samp = [], [], []
for r in (0.0, 1.0, 3.0, 7.0, 15.0, 40.0):
    logits = np.log(pi)[None] + Zs @ (r * U).T - r * r / 2; post = np.exp(logits - logits.max(1, keepdims=True)); post /= post.sum(1, keepdims=True)
    greedy = np.bincount(post.argmax(1), minlength=Kc) / len(Zs)                                         # greedy decoding (what beam search approximates)
    drawn = np.minimum((rg.random((len(Zs), 1)) > post.cumsum(1)).sum(1), Kc - 1); sampled = np.bincount(drawn, minlength=Kc) / len(Zs)   # decoding by sampling from the posterior
    ent_greedy.append(H(greedy)); ent_samp.append(H(sampled)); rows.append((r, *[f"{100 * f:.0f}" for f in greedy], f"{ent_greedy[-1]:.2f}", f"{ent_samp[-1]:.2f}"))
print("training frequencies pi (%):", [f"{100 * p:.0f}" for p in pi], f"| entropy {H(pi):.2f} bits | uniform: {math.log2(Kc):.2f} bits")
print(pd.DataFrame(rows, columns=["latent scale r", *[f"template {k + 1} (%), greedy" for k in range(Kc)], "entropy, greedy", "entropy, sampled"]).to_string(index=False))
print("\nr = 0 (the latent says nothing): GREEDY decoding returns the commonest template every time (entropy 0); decoding by SAMPLING returns the training frequencies (entropy of pi).")
print("as r grows the latent codes the templates apart and the prior draws land in each template's region: greedy decoding then spreads over ALL templates, almost uniformly at the real scale (r ~ 7: the 2 % template is chosen 11 % of the time).")
print("so latent sampling buys diversity, but it over-represents rare templates relative to training frequency: the beam score, not the draw, must say which candidates are plausible (a toy with an exact decoder, not a measurement of Model C).")
assert ent_greedy[0] < 0.01 and ent_greedy[3] > 2.4 and abs(ent_samp[0] - H(pi)) < 0.1

## 6. The authors' saved outputs: funnel, cost, interpolation, Top-$K$
All numbers are the ones saved in the capsule's tutorial notebooks (Model C, `ModelC_Basic_Usage` and `ModelC_Interpolation`).

In [ ]:
funnel = pd.DataFrame([("sampling: 3 x (10 samples x beam 5)", 150, 50, 45, 33, 30), ("test reaction 1: 150 samples x beam 10", 1500, 866, 1199, 482, 166), ("interpolation: midpoint latent, beam 50", 50, 49, 49, 50, 23)],
                      columns=["run", "strings from the beam", "valid", "unique templates (incl. intramolecular variants)", "working (site check passed)", "unique reactant sets"])
print(funnel.to_string(index=False))
print("\nfor test reaction 1: 1,500 strings -> 166 unique reactant sets (11 % of the strings). Without the site check the 1,199 templates would give 5,056 reactant sets (1,464 unique).")
cost = pd.DataFrame([("Model A", "10 molecules x beam 50 (batched)", "2:33", 153 / 10), ("Model B", "10 molecules x beam 50 (batched)", "1:38", 98 / 10), ("Model C", "1 molecule x 150 samples x beam 10 (each of 10 molecules)", "2:30 to 3:24 (mean 2:57)", 177)],
                    columns=["model", "run", "time shown by the progress bars", "seconds per molecule"])
print("\n" + cost.to_string(index=False)); print("(different batch sizes and probably different GPUs; the order of magnitude is the point: the sampling model costs about 10-20 times more per molecule for its 1,500 candidates)")
print("\ninterpolation (authors' notebook): z1 encodes '[C:2]#[CH:1]>>Br-[C:1](-Br)=[CH:2]_*#*[C@@]1(C)CCCCC1=O' (dibromoalkene), z2 encodes '[C:1]#[C:2]>>O=[C:1]-[CH2:2]_*#*[C@@]1(C)CCCCC1=O' (an aldehyde route).")
print("decoding 0.5 z1 + 0.5 z2 gave, among 23 unique reactant sets, mixtures such as the vinyl bromide C[C@]1(C=CBr)CCCCC1=O: halfway between a dibromoalkene and an alkyne-forming aldehyde.")
def wilson(k, n, z=1.96):
    p = k / n; den = 1 + z * z / n; c = (p + z * z / (2 * n)) / den; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den; return c - h, c + h
ks = ["Top-1", "Top-3", "Top-5", "Top-10", "Top-20", "Top-50"]; vals = [10, 30, 30, 40, 60, 60]
print("\nsaved Top-K for the first ten test molecules:", dict(zip(ks, vals)), "| 95 % intervals:", [f"{100 * wilson(v // 10, 10)[0]:.0f}-{100 * wilson(v // 10, 10)[1]:.0f}" for v in vals])
print("the numbers are too small a sample to rank the models (Model A 20-80 %, B 40-90 %, C 10-60 %, on the same ten molecules); the paper compares A and B on the full 95k test set (Figure 3) and uses C for diversity and for reference reactions, not for Top-K.")
assert funnel.shape == (3, 6) and wilson(1, 10)[1] < 0.45

In [ ]:
# C's latent as a distance: the authors use the encoder to find, for a new reaction, the closest training reaction ("reference"); the geometry of that distance is the kernel's
bw_dist = pd.DataFrame([(d_, f"{math.exp(-d_ ** 2 / D / bw):.4f}") for d_ in (1, 7, 20, 50, 71.6, 100)], columns=["Euclidean distance between two latents", "kernel value K = exp(-|d|^2 / (D x 0.64))"])
print(bw_dist.to_string(index=False)); print("\nthe kernel is almost 1 for distances up to about 20 and falls only slowly: with 5,120 dimensions and bandwidth 0.64 the kernel itself barely distinguishes latents at the scale (~7) of the model's latents;")
print("the latent DISTANCE (not the kernel) is what the authors use to find the closest reference reaction (nearest neighbor in the latent space); the m-MMD only regularizes the cloud of latents.")

## 7. Your turn
**Exercise 1.** Change `D` to 512 in Section 3 (a latent of 512 numbers). How large is the penalty at the origin, and how does it change between latent scales 0 and 1?

**Exercise 2.** In Section 5 give template 1 a frequency of 0.9 and the other five 0.02 each. At which latent scale does the sampled distribution become near-uniform?

**Exercise 3.** In Section 4 what latent scale would give a capacity of 100 bits?

**Exercise 4.** The `inverse` layer has 52 million parameters. If the latent had only 1,024 numbers (2 positions) and the condition kept its 10 positions, how many parameters would it have?

**Exercise 5.** With the checkpoint (Part II): sample 10 latents for one product and count how many *unique templates* appear.

In [ ]:
D_ = 512; s_ = 1 / (bw * D_); print("Exercise 1: penalty at the origin with D = 512:", round(1 - (1 + 2 * s_) ** (-D_ / 2), 4))
print("Exercise 3: scale for 100 bits:", round(math.sqrt(2 ** (2 * 100 / D) - 1), 3))
print("Exercise 4:", f"{(2 * 512 + 10 * 512) * (2 * 512) + 2 * 512:,}", "parameters (condition 10 x 512 plus latent 2 x 512 -> 2 x 512)")

## 8. Self-check

In [ ]:
assert sum(real.values()) == 97_287_827 and n_small == formula_small and abs(expected_K(0) - 0.2097) < 0.001
assert abs(D / 2 * math.log2(1 + 0.1 ** 2) - 36.7) < 0.1 and funnel.shape == (3, 6)
print("All checks passed.")

# Part II. Run the authors' code on the real Model C

> **Read this first.** Model C's checkpoint is **not** in the capsule: the README gives a Google Drive link. Part II looks for it, tries to download it (Google Drive often refuses such scripted downloads, in which case the notebook tells you how to fetch it by hand), builds the model with the authors' code, samples templates for a product, post-processes them with RDKit and draws the reactions, then reproduces the interpolation of the authors' notebook. It needs **PyTorch**, **RDKit** and, practically, a **GPU** (the authors' progress bars show about 3 minutes per molecule for 1,500 candidates). **We could not run Part II** (no weights, PyTorch or RDKit where this notebook was written); it follows the authors' Model C tutorial and interpolation notebook, which we read in full. The capsule is MIT-licensed (Copyright 2024 Yu Shee); this notebook contains none of its code. Run the cells one by one.

In [ ]:
!pip install -q rdkit gdown

In [ ]:
import subprocess, importlib
if CAPSULE is None:
    zips = sorted(Path(".").glob("capsule-*.zip")) + sorted(Path("/content").glob("capsule-*.zip"))
    if zips: subprocess.run(["unzip", "-q", "-o", str(zips[0])], check=True); CAPSULE = find_capsule()
HAVE_TORCH_RDKIT = CAPSULE is not None and all(importlib.util.find_spec(m) is not None for m in ("torch", "rdkit"))
print("capsule:", CAPSULE, "| PyTorch and RDKit available:", HAVE_TORCH_RDKIT)
CKPT_NAME = "ModelC_AT_DATA_512DIM_ATOM_NoMASK_5e5_LR_150EP.cpth"; GDRIVE_ID = "1jO732coyBmX4_OeSP5SI4I1EMv8Z7rVk"      # the file named in the capsule's README
def find_checkpoint(max_depth=4, seconds=60):
    """Look for Model C's checkpoint (this name, or any *.cpth with 'ModelC' in it) next to the notebook, in /content, in the capsule and in Google Drive."""
    roots = [Path.cwd(), Path("/content")] + ([CAPSULE / "model_checkpoints", CAPSULE, CAPSULE.parent] if CAPSULE else []) + [Path(p) for p in ("/content/drive/MyDrive", "/content/drive/My Drive")]
    deadline = time.time() + seconds
    for r in dict.fromkeys(x for x in roots if x.is_dir()):
        base = len(r.parts)
        for dirpath, dirnames, files in os.walk(r):
            hit = [f for f in files if f == CKPT_NAME or (f.lower().endswith(".cpth") and "modelc" in f.lower())]
            if hit: return Path(dirpath) / hit[0]
            if time.time() > deadline: return None
            if len(Path(dirpath).parts) - base >= max_depth: dirnames[:] = []
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
    return None
CKPT_C = find_checkpoint()
if CKPT_C is None and HAVE_TORCH_RDKIT:
    try:                                                                                          # Google Drive often refuses scripted downloads of large files; then we fall through to the manual route below
        import gdown
        target = CAPSULE / "model_checkpoints" / CKPT_NAME; target.parent.mkdir(exist_ok=True)
        gdown.download(id=GDRIVE_ID, output=str(target), quiet=False); CKPT_C = target if target.exists() else None
    except Exception as e:
        print("automatic download failed:", type(e).__name__, "(Google Drive quota or permission)"); CKPT_C = None
if CKPT_C is None:
    print("Model C's checkpoint was not found. Get it by hand (it is large, and Google Drive blocks scripted downloads of such files):")
    print("  1. open https://drive.google.com/file/d/" + GDRIVE_ID + "/view in your browser and click Download;")
    print("     (if Drive says the quota is exceeded: choose 'Add shortcut to Drive' or 'Make a copy' from the file's menu, then download the copy from your own Drive)")
    print("  2. put the file ", CKPT_NAME, " into the capsule's 'model_checkpoints' folder (in Drive, or upload it to /content),")
    print("  3. run this cell again: the search above finds it by name. The cells below are skipped until then.")
else:
    print("checkpoint:", CKPT_C, f"{CKPT_C.stat().st_size / 1e6:.0f} MB")

In [ ]:
if HAVE_TORCH_RDKIT and CKPT_C is not None and CKPT_C.exists():
    # a downloaded checkpoint is a pickle from the internet: list what it would import BEFORE loading it
    z = zipfile.ZipFile(CKPT_C); pk = [n for n in z.namelist() if n.endswith("data.pkl")][0]; imports = pickle_globals(z.read(pk)); print("the checkpoint's pickle imports:", imports)
    assert set(imports) <= {"collections.OrderedDict", "torch.FloatStorage", "torch._utils._rebuild_tensor_v2", "torch.HalfStorage", "torch.DoubleStorage", "torch.LongStorage"}, "unexpected imports: do NOT load this file"

In [ ]:
if HAVE_TORCH_RDKIT and CKPT_C is not None and CKPT_C.exists():
    import torch
    from rdkit import Chem, RDLogger
    RDLogger.DisableLog("rdApp.*")
    os.chdir(CAPSULE); sys.path.insert(0, str(CAPSULE))
    from helper import draw_reverse_template, draw_reaction
    from data_AT.data_helper import Tokenizer
    from model_library.modelC.ModelClass import Encoder, Decoder, Conditioner, Seq2Seq
    from model_library.modelC.GenerationFunctions import Batch_Beam_Search, process_beam_results_list, group_translate

    with open("data_AT/data_pickles/ModelA_dictionary.pickle", "rb") as fh: smiledict, invdict = pickle.load(fh)    # Model C uses the same dictionary as Model A
    HID, device, product_length = 512, torch.device("cuda" if torch.cuda.is_available() else "cpu"), 310
    enc = Encoder(len(smiledict) + 1, HID, 6, HID // 32, 2 * HID, 0.1, max_length=350)
    cond_enc = Encoder(len(smiledict) + 1, HID, 6, HID // 32, 2 * HID, 0.1, max_length=product_length)
    conditioner = Conditioner(cond_enc, seq_len=10)
    dec = Decoder(len(invdict), HID, 6, HID // 32, 2 * HID, 0.1, max_length=350)
    modelC = Seq2Seq(enc, dec, conditioner, len(smiledict), len(smiledict), bottle_len=10)
    n_par = sum(p.numel() for p in modelC.parameters() if p.requires_grad); print("parameters:", f"{n_par:,}", "(the authors print 97,287,827)"); assert n_par == 97_287_827
    print(modelC.load_state_dict(torch.load(CKPT_C, map_location=device)["model_state_dict"])); modelC.to(device).eval();

In [ ]:
# sample templates for a product: each call draws `number` latent vectors, each decoded with beam search
if "modelC" in globals():
    tokenizer = Tokenizer("character_consider_atoms")
    TARGET = "C#CC1(C)CCCCC1=O"; target = Chem.MolToSmiles(Chem.MolFromSmiles(TARGET), isomericSmiles=True, canonical=True)
    ids = [smiledict[t] for t in tokenizer.tokenize(target, product=True)]; ids += [len(smiledict)] * (product_length - len(ids)); x = torch.LongTensor(ids).to(device)
    beam = []
    for _ in range(3):
        with torch.no_grad():
            beam += Batch_Beam_Search(x.view(1, -1), modelC, device, smiledict, invdict, max_len=min(349, len(target) * 10), number=10, C=None, beam_size=5)
    beam_list = np.array(beam).reshape((-1, 2)).tolist()
    sorted_check, working_templates, unique_reactant_sets, idx = process_beam_results_list(beam_list, target, Keep_Chirality=True)
    print(f"{len(beam_list)} strings -> {len(unique_reactant_sets)} unique reactant sets (Exercise 5: count the unique templates among them)")

In [ ]:
if "modelC" in globals() and unique_reactant_sets:
    product_mol = Chem.MolFromSmiles(target); n_show = min(4, len(unique_reactant_sets)); fig, axes = plt.subplots(n_show, 2, figsize=(11, 3.2 * n_show), squeeze=False)
    for i in range(n_show):
        img = draw_reaction([Chem.MolFromSmiles(r) for r in unique_reactant_sets[i]], product_mol, 300); tmpl = working_templates[idx[i]]
        axes[i, 0].imshow(img); axes[i, 0].set_title(f"sampled candidate {i + 1}", fontsize=9); axes[i, 1].imshow(draw_reverse_template(tmpl, 220)); axes[i, 1].set_title(tmpl, fontsize=7); axes[i, 0].axis("off"); axes[i, 1].axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
# the authors' interpolation: encode two (template, center-labeled product) strings, take the midpoint of their latents, decode it
if "modelC" in globals():
    prod2 = "C#C[C@@]1(C)CCCCC1=O"; strings = ["[C:2]#[CH:1]>>Br-[C:1](-Br)=[CH:2]_*#*[C@@]1(C)CCCCC1=O", "[C:1]#[C:2]>>O=[C:1]-[CH2:2]_*#*[C@@]1(C)CCCCC1=O"]
    def pad_to(ids, n): return ids + [len(smiledict)] * (n - len(ids))
    prod_t = torch.LongTensor([pad_to([smiledict[t] for t in tokenizer.tokenize(prod2, product=True)], product_length)] * 2).to(device)
    src_t = torch.LongTensor([pad_to([smiledict[t] for t in tokenizer.tokenize(s)], 350) for s in strings]).to(device)
    with torch.no_grad():
        Z = group_translate(src_t.view(-1, 350), prod_t.view(-1, 310), modelC, device, smiledict, invdict, max_len=350, return_Z=True)
    mid = 0.5 * Z[0] + 0.5 * Z[1]; print("latent norms:", [round(float(torch.linalg.norm(z)), 2) for z in (Z[0], Z[1], mid)], "(the midpoint of two roughly orthogonal vectors has norm ~ 1/sqrt(2) of theirs)")
    with torch.no_grad():
        mid_res = Batch_Beam_Search(prod_t[0].view(1, -1), modelC, device, smiledict, invdict, max_len=min(349, len(prod2) * 10), number=1, C=None, beam_size=50, given_Z=mid.view(1, -1).to(device))
    mid_list = np.array(mid_res).reshape((-1, 2)).tolist(); out = process_beam_results_list(mid_list, prod2, Keep_Chirality=True); print(len(out[2]), "unique reactant sets from the midpoint latent")
    for r in list(out[2])[:5]: print("  ", sorted(r))